# Aosta Valley Temperature Prediction — Kriging + ML Pipeline

This notebook covers:
1. Ordinary Kriging (OK) and Universal Kriging (UK) spatial interpolation
2. Keras Tuner hyperparameter search for an MLP
3. Classical ML models (XGBoost, Random Forest, SVR) with proper scaling

**Bug fixes applied in this notebook:**
- **Issue 5** (lines 55-61): `OK.execute` / `UK.execute` wrapped in
  `try/except (LinAlgError, ValueError)` to handle singular covariance
  matrices gracefully.
- **Issue 3** (lines 121-151): After `tuner.search`, the best model is
  retrieved with `tuner.get_best_models(num_models=1)[0]` instead of
  relying on a stale reference.
- **Issue 1** (lines 949-1022): `X_train_scaled` and `X_test_scaled` are
  explicitly defined via `train_test_split` + `StandardScaler` before
  they are consumed by the XGBoost, Random Forest, and SVR blocks.

## 0. Imports and Setup

In [ ]:
import sys
sys.path.append('..')

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import LinAlgError

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data.loader import load_config
from src.data.preprocessor import build_pipeline
from src.models.kriging import run_ordinary_kriging, run_universal_kriging
from src.models.ml_models import prepare_features
from src.utils.metrics import evaluate_model

config = load_config('../config.yaml')
print('Config loaded.')

## 1. Load and Preprocess Data

In [ ]:
station_gdf = build_pipeline(config, base_path='..')
print(f'Dataset shape: {station_gdf.shape}')
station_gdf[['station_id', 'temperature', 'altitude',
             'distance_to_lake', 'distance_to_river', 'hour', 'month']].head()

## 2. Spatial Interpolation — Ordinary and Universal Kriging

### Issue 5 Fix
Calls to `OK.execute` and `UK.execute` internally perform matrix inversion
on the covariance matrix.  When stations are collocated or the grid is
ill-conditioned this raises `numpy.linalg.LinAlgError`.  Both calls are
wrapped in `try/except (LinAlgError, ValueError, RuntimeError)` via the
`run_ordinary_kriging` / `run_universal_kriging` helpers so the pipeline
can continue (returning `None`) instead of crashing.

In [ ]:
# Aggregate to one temperature reading per station (mean across time)
# so we have unique (lon, lat) locations for the variogram fit.
station_snapshot = (
    station_gdf
    .groupby(['station_id', 'longitude', 'latitude'], as_index=False)
    ['temperature'].mean()
    .dropna()
)

x_coords = station_snapshot['longitude'].values
y_coords = station_snapshot['latitude'].values
z_values = station_snapshot['temperature'].values

grid_x = np.linspace(x_coords.min(), x_coords.max(), 50)
grid_y = np.linspace(y_coords.min(), y_coords.max(), 50)

print(f'Stations used for kriging: {len(x_coords)}')
print(f'Grid size: {len(grid_x)} x {len(grid_y)}')

In [ ]:
# ---------------------------------------------------------------------------
# Issue 5 fix (lines 55-61): OK.execute / UK.execute are called inside
# run_ordinary_kriging / run_universal_kriging, which wrap them in
# try/except (LinAlgError, ValueError, RuntimeError).  A singular matrix
# will produce a RuntimeWarning and return (None, None) rather than
# crashing the notebook.
# ---------------------------------------------------------------------------

# --- Ordinary Kriging ---
ok_pred, ok_var = run_ordinary_kriging(
    x_coords, y_coords, z_values,
    grid_x, grid_y,
    variogram_model='linear'
)

if ok_pred is not None:
    print('Ordinary Kriging succeeded.')
    print(f'  Prediction range: [{ok_pred.min():.2f}, {ok_pred.max():.2f}] °C')
else:
    print('Ordinary Kriging failed (LinAlgError or ill-conditioned matrix).')

# --- Universal Kriging ---
uk_pred, uk_var = run_universal_kriging(
    x_coords, y_coords, z_values,
    grid_x, grid_y,
    variogram_model='linear'
)

if uk_pred is not None:
    print('Universal Kriging succeeded.')
    print(f'  Prediction range: [{uk_pred.min():.2f}, {uk_pred.max():.2f}] °C')
else:
    print('Universal Kriging failed (LinAlgError or ill-conditioned matrix).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, title in zip(
    axes,
    [ok_pred, uk_pred],
    ['Ordinary Kriging', 'Universal Kriging'],
):
    if pred is not None:
        im = ax.imshow(
            pred,
            origin='lower',
            extent=[grid_x.min(), grid_x.max(), grid_y.min(), grid_y.max()],
            cmap='RdYlBu_r',
            aspect='auto',
        )
        ax.scatter(x_coords, y_coords, c=z_values,
                   cmap='RdYlBu_r', edgecolors='k', s=40, zorder=5)
        fig.colorbar(im, ax=ax, label='Temperature (°C)')
    else:
        ax.text(0.5, 0.5, 'Kriging failed', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='red')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

## 3. Keras Tuner MLP

### Issue 3 Fix
After `tuner.search`, **only** `get_best_hyperparameters()` was called in
the original code.  This left `model` pointing at a stale, incorrectly
configured object rather than the best-trained model.  The fix is:

```python
# WRONG (original)
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
y_pred = model.predict(X_test_scaled)   # model is stale!

# CORRECT (fixed)
best_model = tuner.get_best_models(num_models=1)[0]
y_pred = best_model.predict(X_test_scaled)
```

In [ ]:
try:
    import tensorflow as tf
    import keras_tuner as kt
    _TF_AVAILABLE = True
except ImportError:
    _TF_AVAILABLE = False
    print('TensorFlow / keras-tuner not installed — skipping Keras Tuner cell.')

In [ ]:
if _TF_AVAILABLE:
    # -----------------------------------------------------------------------
    # Prepare features: drop geometry-only columns and the target, then split.
    # -----------------------------------------------------------------------
    X_train, X_test, y_train, y_test = prepare_features(station_gdf, config)

    # Convert to plain numpy arrays for TensorFlow
    numeric_cols = X_train.select_dtypes(include='number').columns.tolist()
    X_train_np = X_train[numeric_cols].values.astype(np.float32)
    X_test_np  = X_test[numeric_cols].values.astype(np.float32)
    y_train_np = y_train.values.astype(np.float32)
    y_test_np  = y_test.values.astype(np.float32)

    # Scale features — fit on training data only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_np)
    X_test_scaled  = scaler.transform(X_test_np)

    n_features = X_train_scaled.shape[1]

    def build_model(hp):
        model = tf.keras.Sequential()
        model.add(tf.keras.layers.InputLayer(input_shape=(n_features,)))
        for i in range(hp.Int('num_layers', 1, 3)):
            units = hp.Choice(f'units_{i}', [32, 64, 128])
            model.add(tf.keras.layers.Dense(units, activation='relu'))
            model.add(tf.keras.layers.Dropout(
                hp.Float(f'dropout_{i}', 0.0, 0.4, step=0.1)
            ))
        model.add(tf.keras.layers.Dense(1))
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=hp.Choice('lr', [1e-2, 1e-3, 1e-4])
            ),
            loss='mse',
        )
        return model

    tuner = kt.RandomSearch(
        build_model,
        objective='val_loss',
        max_trials=5,
        executions_per_trial=1,
        directory='kt_dir',
        project_name='krigmain_mlp',
        overwrite=True,
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    tuner.search(
        X_train_scaled, y_train_np,
        epochs=30,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0,
    )

    # -------------------------------------------------------------------
    # Issue 3 fix (lines 121-151): retrieve the best FITTED model.
    # get_best_models() returns models that have already been trained
    # during tuner.search, unlike get_best_hyperparameters() which only
    # returns the config values and leaves any separate model object
    # in whatever state it was last left in (typically untrained or
    # trained on a different hyper-parameter configuration).
    # -------------------------------------------------------------------
    best_model = tuner.get_best_models(num_models=1)[0]
    best_model.build(input_shape=(None, n_features))

    y_pred_kt = best_model.predict(X_test_scaled, verbose=0).flatten()
    kt_result = evaluate_model(y_test_np, y_pred_kt, model_name='KerasTunerMLP')
    print('Best HPs:', tuner.get_best_hyperparameters(num_trials=1)[0].values)
    print(kt_result)

## 4. Feature Engineering and Train / Test Split

### Issue 1 Fix
The original notebook referenced `X_train_scaled` and `X_test_scaled` in
the XGBoost, Random Forest, and SVR blocks (approx. lines 949–1022) but
never defined them, causing an immediate `NameError`.  The cell below
defines them explicitly via `train_test_split` + `StandardScaler` **before**
those model blocks.

In [ ]:
# -------------------------------------------------------------------------
# Issue 1 fix (lines 949-1022): define X_train_scaled / X_test_scaled
# BEFORE they are used in the XGBoost, Random Forest, and SVR blocks.
# -------------------------------------------------------------------------

# Step 1 — Build feature matrix and target vector
target_col = config['features']['target']          # e.g. 'temperature'
drop_cols  = config['features']['drop_cols']       # e.g. geometry cols

y_full = station_gdf[target_col].dropna()
X_full = station_gdf.drop(columns=drop_cols).loc[y_full.index]

# Keep only numeric columns for the sklearn models below
X_numeric = X_full.select_dtypes(include='number')
X_numeric = X_numeric.apply(lambda col: col.fillna(col.median()))

# Step 2 — Train / test split (80 / 20, stratified by nothing — regression)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_numeric, y_full,
    test_size=0.2,
    random_state=42,
)

# Step 3 — Scale: fit ONLY on the training partition
feature_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_train_raw)   # defined here
X_test_scaled  = feature_scaler.transform(X_test_raw)        # defined here

print(f'X_train_scaled : {X_train_scaled.shape}')
print(f'X_test_scaled  : {X_test_scaled.shape}')

## 5. XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor

xgb_cfg = config['models']['xgboost']
xgb_model = XGBRegressor(
    n_estimators=xgb_cfg['n_estimators'],
    learning_rate=xgb_cfg['learning_rate'],
    max_depth=xgb_cfg['max_depth'],
    random_state=xgb_cfg['random_state'],
    verbosity=0,
)

# X_train_scaled and X_test_scaled are now defined (Issue 1 fix above)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)

xgb_result = evaluate_model(y_test, y_pred_xgb, model_name='XGBoost')
print(xgb_result)

## 6. Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_cfg = config['models']['random_forest']
rf_model = RandomForestRegressor(
    n_estimators=rf_cfg['n_estimators'],
    random_state=rf_cfg['random_state'],
    n_jobs=-1,
)

# X_train_scaled and X_test_scaled are defined (Issue 1 fix above)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

rf_result = evaluate_model(y_test, y_pred_rf, model_name='RandomForest')
print(rf_result)

## 7. Support Vector Regressor (SVR)

In [ ]:
from sklearn.svm import SVR

svr_cfg = config['models']['svr']

# SVR scales poorly with N — subsample the training set
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train_scaled, y_train,
    train_size=0.1,
    random_state=42,
)

svm_model = SVR(kernel=svr_cfg['kernel'], C=svr_cfg['C'])

# X_train_scaled and X_test_scaled are defined (Issue 1 fix above)
svm_model.fit(X_train_sub, y_train_sub)
y_pred_svm = svm_model.predict(X_test_scaled)

svm_result = evaluate_model(y_test, y_pred_svm, model_name='SVR')
print(svm_result)

## 8. Model Comparison Summary

In [ ]:
results = [xgb_result, rf_result, svm_result]
summary_df = (
    pd.DataFrame(results)
    .set_index('model')
    .sort_values('r2', ascending=False)
)
print(summary_df.to_string())
summary_df.style.format({'mse': '{:.4f}', 'mae': '{:.4f}', 'r2': '{:.4f}'}).background_gradient(subset=['r2'], cmap='RdYlGn')